## Prerequisite Library

In [ ]:
!pip install python-dotenv sqlalchemy psycopg2-binary pandas

In [ ]:
import pandas as pd
import numpy as np
import re

# Load data
df = pd.read_csv("samsung_phones_specs.csv")

print("Original Data Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample Data:")
print(df.head())
print("\nData Types:")
print(df.dtypes)

# Data Normalization

We will normalize the Samsung phones CSV into multiple related tables:

1. **phones** - Main product table with basic info
2. **brands** - Brand lookup table  
3. **operating_systems** - OS lookup table
4. **cpu_models** - CPU model lookup table
5. **phone_specs** - Technical specifications (RAM, CPU speed)

In [ ]:
# Add random prices between $300 and $1000 where price is missing
# Generate random prices only for missing values
missing_price_mask = df['price'].isna()
random_prices = np.random.randint(300, 1001, size=missing_price_mask.sum())
df.loc[missing_price_mask, 'price'] = random_prices

print("Updated DataFrame with random prices:")
print(df[['name', 'price']].head(10))
print(f"\nPrice statistics:")
print(df['price'].describe())

In [ ]:

# df.to_csv("samsung_phones_specs_with_prices_with_random_prices.csv", index=False)

In [ ]:
# Data Cleaning - Handle missing values and standardize data
df_clean = df.copy()

# Fill missing brand values
df_clean['brand'] = df_clean['brand'].fillna('Unknown')

# Fill missing OS values
df_clean['operating_system'] = df_clean['operating_system'].fillna('Not Specified')

# Fill missing RAM values
df_clean['ram'] = df_clean['ram'].fillna('Not Specified')

# Fill missing CPU model values
df_clean['cpu_model'] = df_clean['cpu_model'].fillna('Not Specified')

# Fill missing CPU speed values
df_clean['cpu_speed'] = df_clean['cpu_speed'].fillna('Not Specified')

# Clean ratings_count - extract numeric value
def clean_ratings(val):
    if pd.isna(val):
        return 0
    # Remove parentheses and commas, convert to int
    cleaned = str(val).replace('(', '').replace(')', '').replace(',', '')
    try:
        return int(cleaned)
    except:
        return 0

df_clean['ratings_count'] = df_clean['ratings_count'].apply(clean_ratings)

print("Cleaned Data Sample:")
print(df_clean.head())
print("\nMissing Values After Cleaning:")
print(df_clean.isnull().sum())

In [ ]:
# ===========================================
# TABLE 1: Brands (Lookup Table)
# ===========================================
brands = df_clean['brand'].unique()
brands_df = pd.DataFrame({
    'brand_id': range(1, len(brands) + 1),
    'brand_name': brands
})

print("BRANDS TABLE:")
print(brands_df)
print(f"\nTotal unique brands: {len(brands_df)}")

In [ ]:
# ===========================================
# TABLE 2: Operating Systems (Lookup Table)
# ===========================================
operating_systems = df_clean['operating_system'].unique()
os_df = pd.DataFrame({
    'os_id': range(1, len(operating_systems) + 1),
    'os_name': operating_systems
})

print("OPERATING SYSTEMS TABLE:")
print(os_df)
print(f"\nTotal unique operating systems: {len(os_df)}")

In [ ]:
# ===========================================
# TABLE 3: CPU Models (Lookup Table)
# ===========================================
cpu_models = df_clean['cpu_model'].unique()
cpu_df = pd.DataFrame({
    'cpu_id': range(1, len(cpu_models) + 1),
    'cpu_model': cpu_models
})

print("CPU MODELS TABLE:")
print(cpu_df)
print(f"\nTotal unique CPU models: {len(cpu_df)}")

In [ ]:
# ===========================================
# TABLE 4: Phones (Main Product Table)
# ===========================================

# Create mapping dictionaries for foreign keys
brand_mapping = dict(zip(brands_df['brand_name'], brands_df['brand_id']))
os_mapping = dict(zip(os_df['os_name'], os_df['os_id']))
cpu_mapping = dict(zip(cpu_df['cpu_model'], cpu_df['cpu_id']))

# Create phones table with foreign keys
phones_df = pd.DataFrame({
    'phone_id': range(1, len(df_clean) + 1),
    'name': df_clean['name'],
    'price': df_clean['price'],
    'brand_id': df_clean['brand'].map(brand_mapping),
    'os_id': df_clean['operating_system'].map(os_mapping),
    'ratings_count': df_clean['ratings_count'],
    'url': df_clean['url']
})

print("PHONES TABLE (Main Table):")
print(phones_df.head(10))
print(f"\nTotal phones: {len(phones_df)}")

In [ ]:
# ===========================================
# TABLE 5: Phone Specifications (Technical Details)
# ===========================================

phone_specs_df = pd.DataFrame({
    'spec_id': range(1, len(df_clean) + 1),
    'phone_id': range(1, len(df_clean) + 1),
    'ram': df_clean['ram'],
    'cpu_id': df_clean['cpu_model'].map(cpu_mapping),
    'cpu_speed': df_clean['cpu_speed']
})

print("PHONE SPECIFICATIONS TABLE:")
print(phone_specs_df.head(10))
print(f"\nTotal specs records: {len(phone_specs_df)}")

In [ ]:
# ===========================================
# Save Normalized Tables to CSV Files
# ===========================================

# Save all normalized tables
brands_df.to_csv('dataset/brands.csv', index=False)
os_df.to_csv('dataset/operating_systems.csv', index=False)
cpu_df.to_csv('dataset/cpu_models.csv', index=False)
phones_df.to_csv('dataset/phones.csv', index=False)
phone_specs_df.to_csv('dataset/phone_specs.csv', index=False)
print("✅ All normalized tables saved successfully!")
print("\nFiles created:")
print("  1. normalized_brands.csv")
print("  2. normalized_operating_systems.csv")
print("  3. normalized_cpu_models.csv")
print("  4. normalized_phones.csv")
print("  5. normalized_phone_specs.csv")

In [ ]:
# ===========================================
# Save Normalized Tables to PostgreSQL Database
# ===========================================
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Load environment variables from .env file
load_dotenv()

# Get database connection parameters from .env
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create PostgreSQL connection string
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# Save all normalized tables to PostgreSQL
brands_df.to_sql('brands', engine, if_exists='replace', index=False)
os_df.to_sql('operating_systems', engine, if_exists='replace', index=False)
cpu_df.to_sql('cpu_models', engine, if_exists='replace', index=False)
phones_df.to_sql('phones', engine, if_exists='replace', index=False)
phone_specs_df.to_sql('phone_specs', engine, if_exists='replace', index=False)

print("✅ All normalized tables saved to PostgreSQL successfully!")
print("\nTables created:")
print("  1. brands")
print("  2. operating_systems")
print("  3. cpu_models")
print("  4. phones")
print("  5. phone_specs")